# YOLO Radio Astronomy Source Detection

This notebook presents the second-round SKA-like radio source detector: spatially separated validation, small-source-aware tiling, YOLO metrics, and flux-dependent scientific evaluation.


In [1]:
from pathlib import Path
import sys

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT))

from src.catalog import read_sdc1_catalog, summarize_catalog
from src.config import load_yaml
from src.paths import resolve_project_path
from src.reporting import read_training_results, latest_metric_snapshot
from src.runtime import configure_project_cache
from src.visualization import draw_yolo_boxes, plot_tile_grid


D:\Anaconda\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


TypeError: unsupported operand type(s) for |: 'type' and 'type'

## 1. Data and Task

The input is SKA SDC1 simulated continuum imaging. Round 2 uses class-agnostic source detection and retains catalogue classes for separate SS-AGN, FS-AGN, and SFG completeness analysis.


In [ ]:
CACHE_ROOT = configure_project_cache()
data_cfg = load_yaml("configs/sdc1_b1_round2.yaml")
catalog = read_sdc1_catalog(resolve_project_path(data_cfg["catalog"]))
{"cache_root": CACHE_ROOT.relative_to(PROJECT), **summarize_catalog(catalog)}


## 2. Prepare YOLO Tiles

The prepared Round 2 dataset is already available. Regeneration is disabled by default and all generated tiles remain outside Git.


In [ ]:
from src.prepare_dataset import prepare_dataset

RUN_PREPARATION = False
if RUN_PREPARATION:
    prepare_dataset("configs/sdc1_b1_round2.yaml")
else:
    print("Dataset regeneration is disabled.")


## 3. Inspect Training Tiles

The 320-pixel tiles are enlarged to 640 pixels during training so that compact sources occupy more network pixels. Training and validation tiles come from non-overlapping sky regions.


In [ ]:
dataset_dir = resolve_project_path(data_cfg["output_dir"])
image_dir = dataset_dir / "images" / "train"
label_dir = dataset_dir / "labels" / "train"
images = sorted(image_dir.glob("*.png"))[:6]
items = []
for image_path in images:
    label_path = label_dir / f"{image_path.stem}.txt"
    items.append((draw_yolo_boxes(image_path, label_path, data_cfg["classes"]), image_path.name))
plot_tile_grid(items, columns=3, title="Training tiles with catalogue-derived boxes") if items else "Run dataset preparation first."


## 4. Train YOLO

Training is controlled explicitly by the switch below. It defaults to `False`; change it to `True` only when you are ready to use the GPU. Runtime caches and temporary files are routed to the project drive.


In [ ]:
from src.train import train

RUN_TRAINING = False
if RUN_TRAINING:
    train("configs/train_round2.yaml")
else:
    print("Training is disabled. Set RUN_TRAINING = True to start Round 2.")


## 5. Training Curves and Metrics

For object detection, the relevant metrics are loss curves, Precision, Recall, F1, mAP50, and mAP50-95. Accuracy alone is not meaningful for this sparse detection problem.


In [ ]:
run_dir = resolve_project_path("outputs/runs/yolo_sdc1_round2")
results = read_training_results(run_dir)
latest_metric_snapshot(results)


In [2]:
if not results.empty:
    axes = results.plot(
        x="epoch",
        y=[col for col in results.columns if "loss" in col or "mAP50" in col or "precision" in col or "recall" in col],
        subplots=True,
        layout=(2, 3),
        figsize=(15, 7),
        legend=True,
    )


NameError: name 'results' is not defined

## 6. Evaluate and Inspect Predictions

Evaluation is also opt-in. Standard YOLO metrics are complemented by deduplicated completeness versus catalogue flux, reliability versus confidence, and completeness by astrophysical population.


In [ ]:
from src.evaluate import evaluate

RUN_EVALUATION = False
if RUN_EVALUATION:
    evaluate("configs/train_round2.yaml", "outputs/runs/yolo_sdc1_round2/weights/best.pt", "round2_yolo_summary.json")
else:
    print("Evaluation is disabled until a trained checkpoint is available.")


## 7. Success and Failure Cases

The scientific evaluator selects representative success and failure tiles automatically. Green/cyan boxes are matched catalogue/prediction boxes; red boxes are missed sources and yellow boxes are false detections.


In [ ]:
from src.scientific_evaluation import evaluate_scientifically

RUN_SCIENTIFIC_EVALUATION = False
if RUN_SCIENTIFIC_EVALUATION:
    evaluate_scientifically("configs/train_round2.yaml", "outputs/runs/yolo_sdc1_round2/weights/best.pt")
else:
    print("Scientific evaluation is disabled until training is complete.")


In [ ]:
pred_dir = resolve_project_path("outputs/predictions/round2_examples")
pred_images = sorted(pred_dir.glob("*.png"))[:6]
plot_tile_grid(pred_images, columns=3, title="Validation prediction examples") if pred_images else "Run prediction export first."


## 8. Scientific Notes

Round 2 separates the engineering detector from the scientific interpretation: YOLO learns one source class, while catalogue metadata measures population- and flux-dependent selection effects. This avoids presenting morphology-only image classification as a physical source classifier.
